<a href="https://colab.research.google.com/github/falvojr/santander-dev-week-2023/blob/master/SantanderDevWeek2023.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚨 A API Está Fora do Ar? Sem Problemas!

A API utilizada neste projeto (`sdw-2023-prd.up.railway.app`) era um ambiente público de demonstração e foi descontinuada. **Isso não impede você de concluir este desafio** 🚀

Como reforçado nas aulas, o objetivo principal aqui é compreender o fluxo **ETL (Extração, Transformação e Carregamento)**. Se a API não estiver respondendo, você pode substituir a etapa de **Extract** e ajustar a **Load** sem prejuízo ao aprendizado.


## Ajustando a Etapa de Extract

### 🟢 Opção 1: Mais Simples (Dados Direto no Código)

Ideal se você quiser focar exclusivamente na lógica e no uso de IA, sem depender de arquivos externos.

```python
# Simula a extração de dados (substituindo o GET da API)
users = [
    {'id': 1, 'name': 'Naruto', 'news': []},
    {'id': 2, 'name': 'Hinata', 'news': []}
]
```

### 🟡 Opção 2: Mais Completa (Leitura de Arquivo)

Você pode adaptar o arquivo `SDW2023.csv`, incluindo a coluna `name`, e usá-lo como fonte de dados.

```python
import pandas as pd

# Lê o CSV e converte para uma lista de dicionários
users = pd.read_csv('SDW2023.csv').to_dict(orient='records')

# Garante a estrutura esperada para a etapa de Transformação
for user in users:
    user['news'] = []
```

## E a Etapa de Load?

Como a API está indisponível, a chamada `PUT` não funcionará. Para finalizar o Lab, basta salvar o resultado localmente (em um arquivo CSV, JSON ou mesmo exibindo no console).

O mais importante é demonstrar que você entende como os dados percorrem todas as etapas do **ETL**, independentemente da ferramenta ou fonte utilizada 😉

# Santander Dev Week 2023 (ETL com Python)

**Contexto:** Você é um cientista de dados no Santander e recebeu a tarefa de envolver seus clientes de maneira mais personalizada. Seu objetivo é usar o poder da IA Generativa para criar mensagens de marketing personalizadas que serão entregues a cada cliente.

**Condições do Problema:**

1. Você recebeu uma planilha simples, em formato CSV ('SDW2023.csv'), com uma lista de IDs de usuário do banco:
  ```
  UserID
  1
  2
  3
  4
  5
  ```
2. Seu trabalho é consumir o endpoint `GET https://sdw-2023-prd.up.railway.app/users/{id}` (API da Santander Dev Week 2023) para obter os dados de cada cliente.
3. Depois de obter os dados dos clientes, você vai usar a API do ChatGPT (OpenAI) para gerar uma mensagem de marketing personalizada para cada cliente. Essa mensagem deve enfatizar a importância dos investimentos.
4. Uma vez que a mensagem para cada cliente esteja pronta, você vai enviar essas informações de volta para a API, atualizando a lista de "news" de cada usuário usando o endpoint `PUT https://sdw-2023-prd.up.railway.app/users/{id}`.



In [ ]:
# Utilize sua própria URL se quiser ;)
# Repositório da API: https://github.com/digitalinnovationone/santander-dev-week-2023-api
sdw2023_api_url = 'https://sdw-2023-prd.up.railway.app'

## **E**xtract

Como a API pública da Santander Dev Week 2023 está indisponível, a extração será feita a partir do arquivo local `SDW2023.csv`. O arquivo contém as colunas `UserID` e `name`, permitindo montar a estrutura mínima de usuários para a etapa de transformação sem depender do endpoint externo.


In [ ]:
import pandas as pd

# Lê o CSV do Plano B. Ele deve conter, no mínimo, as colunas UserID e name.
df = pd.read_csv('SDW2023.csv')

required_columns = {'UserID', 'name'}
missing_columns = required_columns - set(df.columns)
if missing_columns:
  raise ValueError(f'O arquivo SDW2023.csv está sem as colunas obrigatórias: {missing_columns}')

# Mantém compatibilidade com a estrutura original esperada pelo desafio.
users = df.rename(columns={'UserID': 'id'})[['id', 'name']].to_dict(orient='records')

# Garante a estrutura esperada pela etapa de transformação.
for user in users:
  user['news'] = []

print(users)


In [ ]:
import json

# A API pública original está fora do ar; portanto, os usuários já foram extraídos do CSV.
print(json.dumps(users, ensure_ascii=False, indent=2))


## **T**ransform

Utilize a API do OpenAI GPT-4 para gerar uma mensagem de marketing personalizada para cada usuário.

In [ ]:
!pip install openai

In [ ]:
import os

# Se você tiver uma chave da OpenAI, defina a variável de ambiente OPENAI_API_KEY.
# Caso contrário, o notebook usará uma função local simples para gerar mensagens de exemplo.
openai_api_key = os.getenv('OPENAI_API_KEY', 'TODO')


In [ ]:
def generate_ai_news(user):
  prompt = f"Crie uma mensagem para {user['name']} sobre a importância dos investimentos (máximo de 100 caracteres)"

  if openai_api_key and openai_api_key != 'TODO':
    try:
      from openai import OpenAI
      client = OpenAI(api_key=openai_api_key)
      completion = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
          {
            'role': 'system',
            'content': 'Você é um especialista em marketing bancário.'
          },
          {
            'role': 'user',
            'content': prompt
          }
        ]
      )
      return completion.choices[0].message.content.strip('\"')
    except Exception as error:
      print(f'Não foi possível usar a OpenAI agora ({error}). Usando mensagem local de fallback.')

  return f"{user['name']}, investir hoje é cuidar do seu futuro financeiro. Comece agora!"

for user in users:
  news = generate_ai_news(user)
  print(news)
  user['news'].append({
      'icon': 'https://digitalinnovationone.github.io/santander-dev-week-2023-api/icons/credit.svg',
      'description': news
  })


## **L**oad

Como a API está indisponível, a etapa de carga será concluída salvando o resultado localmente. O arquivo `SDW2023_resultado.csv` registra cada usuário e a mensagem de marketing gerada.


In [ ]:
# Salva localmente o resultado do ETL, substituindo o PUT na API indisponível.
result_rows = []

for user in users:
  descriptions = [item['description'] for item in user.get('news', [])]
  result_rows.append({
      'UserID': user['id'],
      'name': user['name'],
      'news': ' | '.join(descriptions)
  })

result_df = pd.DataFrame(result_rows)
result_df.to_csv('SDW2023_resultado.csv', index=False)

print('Arquivo SDW2023_resultado.csv gerado com sucesso!')
display(result_df)
